In [1]:
version = '6.6'

In [2]:
from ast import literal_eval
from bs4 import BeautifulSoup
import codecs
import pandas as pd
import re
import requests

In [3]:
def get_page_soup(url):
    response = requests.get(url)
    if response.status_code == 200:
        return BeautifulSoup(response.text, 'html.parser')

# Decode JS-style escapes (\x3d, \/)
def clean_url(raw: str) -> str:
    return codecs.decode(raw.replace(r'\/', '/'), 'unicode-escape')

def split_string_by_digits(numbered_string_to_split, pattern=r'(\d+)\.'):
    matches = list(re.finditer(pattern, numbered_string_to_split))
    result = {}

    for i in range(len(matches)):
        start = matches[i].start()
        end = matches[i+1].start() if i+1 < len(matches) else len(numbered_string_to_split)
        key = int(matches[i].group(1))
        value = numbered_string_to_split[start:end].strip()
        
        if "~=" in value:
            result[key] = [item.strip() for item in value.split("~=")]
        else:
            result[key] = value[len(matches[i].group(0)):].strip()

    return result


def split_mainstats_string(mainstats_string):
    keywords = ['Sands', 'Goblet', 'Circlet']
    result = {}

    for i, key in enumerate(keywords):
        # Locate the start of the current keyword
        key_start = mainstats_string.find(key)
        if key_start == -1:
            continue  # If the keyword is not found, skip it

        # Locate the end of the section
        if i < len(keywords) - 1:
            # For Sands and Goblet, stop at the next keyword
            next_key_start = mainstats_string.find(keywords[i + 1])
            section = mainstats_string[key_start + len(key) + 1:next_key_start].strip()
        else:
            # For Circlet, stop at the first occurrence of "*", "Check Notes", or the end of the line
            end_marker_positions = [
                mainstats_string.find("*", key_start),
                mainstats_string.find("Check Notes", key_start)
            ]
            # Filter out -1 values and get the earliest valid marker
            end_marker_positions = [pos for pos in end_marker_positions if pos != -1]
            end_marker = min(end_marker_positions) if end_marker_positions else -1
            section = mainstats_string[key_start + len(key) + 1:end_marker].strip() if end_marker != -1 else mainstats_string[key_start + len(key) + 1:].strip()

        # Clean up the section and extract values
        section = section.lstrip('-').strip()  # Remove leading dashes and whitespace

        # Split values if necessary
        values = [v.strip() for v in section.split('/') if v]
        result[key] = values if len(values) > 1 else values[0]

    return result

def normalize_df(df, col_name):
    df[col_name] = df[col_name].map(lambda value: literal_eval(value))
    normalized_data = pd.json_normalize(df[col_name]).add_prefix(f'{col_name}_')
    df = pd.concat([df, normalized_data], axis=1)
    df = df.drop(col_name, axis=1)
    return df

In [4]:
# Get list of characters from genshin.gg
url = 'https://genshin.gg/characters'
character_soup = get_page_soup(url)
char_list = []
char_object = character_soup.find_all('h2', class_='character-name')
for character in char_object:
    char_list.append(character.string.upper())

# Append additional character names where the genshin.gg name doesn't match the character title in the build sheet
char_list.extend([
    'KAMISATO AYAKA',
    'KAMISATO AYATO',
    'ARATAKI ITTO',
    'KAEDEHARA KAZUHA',
    'SANGONOMIYA KOKOMI',
    'RAIDEN SHOGUN',
    'TRAVELER',
    'SHIKANOIN HEIZOU',
    'KUJOU SARA'
])

In [5]:
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRq-sQxkvdbvaJtQAGG6iVz2q2UN9FCKZ8Mkyis87QHFptcOU3ViLh0_PJyMxFSgwJZrd10kbYpQFl1/pubhtml#'
content_soup = get_page_soup(url)

element_list = [
    'Pyro',
    'Hydro',
    'Electro',
    'Dendro',
    'Cryo',
    'Anemo',
    'Geo'
]

# Extract urls for each element's tab
scripts = content_soup.find_all("script")
pattern = re.compile(r'name:\s*"([^"]+)"\s*,\s*pageUrl:\s*"([^"]+)"')
element_urls = []
for script in scripts:
    if script.string and 'items.push' in script.string:
        matches = pattern.findall(script.string)
        for name, url in matches:
            if name.strip() in element_list:
                element_urls.append({'name': name.strip(), 'pageUrl': clean_url(url)})

In [6]:
build_name_row = None
character_list = []

# The exception list below denotes characters with abnormal table header formatting before their builds
abnormal_header_characters = [
    'MAVUIKA',
    'WANDERER',
    'DURIN',
    'NICOLE'
]

# The exception list below denotes rows where a character has multiple roles whose builds share some or all relevant attributes (artifacts, main stats, substats)
# By default, we assume all the same attributes as the previous build.
exception_list_merged_cells = {
    'XIANGLING': 'OFF-FIELD DPS✩',
    'XINGQIU': 'LOW ENERGY REQUIREMENT(OFF-FIELD DPS)✩',
    'YELAN': 'LOW ENERGY REQUIREMENT(OFF-FIELD DPS)✩',
    'BEIDOU': 'OFF-FIELD DPS ✩',
    'SETHOS': 'CHARGED ATTACK DPS✩',
    'LAUMA': 'BUFF SUPPORT(LOW-ENERGY REQUIREMENT) ✩',
    'ESCOFFIER': 'LOW ENERGY REQUIREMENT (OFF-FIELD DPS)✩',
    'LAN YAN': 'DRIVER',
    'WANDERER': 'DPS (WITH EXTERNAL ATK BUFFS)✩',
    'XIANYUN': 'LOW ENERGY REQUIREMENTBUFF & HEAL SUPPORT✩'
}

for element_url in element_urls:
    content_soup = get_page_soup(element_url['pageUrl'])
    table = content_soup.find('table', class_='waffle')
    if table:
        container = table.find('tbody')

    character_name = None

    for row_index, row in enumerate(container):
        for column_index, column in enumerate(row.contents):
            # Search for character name rows
            if column.text in char_list:
                character_row = row_index
                character_name = column.text

                # Characters typically have 4 rows between the name and the first build:
                # - one additional row for the name since most names use merged double-rows
                # - one row for the last update version
                # - one row for the various column headers
                # - one row for column subheaders (WEAPON and ARTIFACT are grouped under EQUIPMENT, MAIN STATS and SUBSTATS are grouped under ARTIFACT STATS
                # As of 2026-05-26, the exceptions are Mavuika, Durin, and Nicole, who only use one row for their names, and Wanderer, who is missing the column header row for some reason
                if character_name in abnormal_header_characters:
                    build_name_row = character_row + 4
                else:
                    build_name_row = character_row + 5
            # Once we reach the NOTES row, we stop
            elif column.text[:5].upper() == 'NOTES':
                build_name_row = None
                break
            elif row_index == build_name_row:
                column_text = column.text
                # Wanderer's formatting is all over the place, so aside from missing a header row that other characters have,
                # he also seems to have an additional column that shifts the column indices of his first build one higher compared to everyone else,
                # but not those of his second.
                if character_name == 'WANDERER' and not build_name == 'DPS (WITH EXTERNAL ATK BUFFS)✩':
                    match column_index:
                        case 2:
                            # For his first build, the build_name will be set to empty by this case, then set to the actual value in the next case
                            # For his second build, the build_name will be set by this case.
                            # The build_name check a few lines before will then prevent the other cases from being hit, causing the same attribute values to be inherited from the first build.
                            build_name = column_text.lstrip()
                        # The remaining cases are only triggered for his first build
                        case 3:
                            build_name = column_text.lstrip()
                        case 4:
                            weapons = split_string_by_digits(column_text)
                        case 5:
                            artifacts = split_string_by_digits(column_text)
                        case 6:
                            main_stats = split_mainstats_string(column_text)
                        case 7:
                            substats = split_string_by_digits(column_text)
                elif not (character_name in exception_list_merged_cells.keys() and exception_list_merged_cells[character_name] == build_name):
                    match column_index:
                        case 2:
                            build_name = column_text.lstrip()
                        case 3:
                            weapons = split_string_by_digits(column_text)
                        case 4:
                            artifacts = split_string_by_digits(column_text)
                        case 5:
                            main_stats = split_mainstats_string(column_text)
                        case 6:
                            substats = split_string_by_digits(column_text)
                else:
                    # We perform special handling for characters whose builds don't use all the same attributes as the previous build here:
                    match column_index:
                        case 3:
                            # Strictly speaking, while some of the characters in the exception list have differing weapons between builds,
                            # we don't really care about weapons for sake of the spreadsheet, so we don't bother to actually perform any special handling
                            pass
                        case 4:
                            # Characters with different artifact sets between builds
                            if character_name == 'SETHOS' and build_name == 'CHARGED ATTACK DPS✩':
                                artifacts = split_string_by_digits(column_text)
                        case 5:
                            # Characters with different main stats between builds
                            if ((character_name == 'LAUMA' and build_name == 'BUFF SUPPORT(LOW-ENERGY REQUIREMENT) ✩')
                                    or (character_name == 'ESCOFFIER' and build_name == 'LOW ENERGY REQUIREMENT (OFF-FIELD DPS)✩')
                                    or (character_name == 'LAN YAN' and build_name == 'DRIVER')):
                                main_stats = split_mainstats_string(column_text)
                        case 6:
                            # Characters with different substats between builds
                            pass
                
        # By this point, we should have enumerated through all the columns, so if we're in a build_name_row, we can save the results
        if row_index == build_name_row:
            character_dict = {
                'character_name': character_name,
                'element': element_url['name'],
                'build_name': build_name,
                'weapons': weapons,
                'artifacts': artifacts,
                'main_stats': main_stats,
                'substats': substats
            }
            character_list.append(character_dict)
            build_name_row += 1

In [7]:
df = pd.DataFrame.from_dict(character_list)
df.to_csv(f'gi_rsh_output_{version}.csv', index=False)
df

,character_name,element,build_name,weapons,artifacts,main_stats,substats
0,AMBER,Pyro,MELT DPS,"{1: 'The First Great Magic (5✩)', 2: 'Aqua Sim...","{1: 'Shimenawa's Reminiscence (4)', 2: 'Wander...","{'Sands': ['ATK%', 'Elemental Mastery'], 'Gobl...","{1: 'Crit DMG', 2: 'ATK%', 3: 'Elemental Maste..."
1,AMBER,Pyro,BUFF SUPPORT✩,"{1: 'Elegy for the End (5✩)', 2: 'Favonius War...","{1: 'Noblesse Oblige (4)', 2: 'Instructor (4)*...","{'Sands': ['Energy Recharge', 'ATK%'], 'Goblet...","{1: 'Energy Recharge', 2: 'Crit Rate / DMG', 3..."
2,XIANGLING,Pyro,OFF-FIELD VAPORIZE DPS ✩,"{1: 'Staff of the Scarlet Sands (5✩)', 2: 'Lum...","{1: 'Emblem of Severed Fate (4)', 2: 'Crimson ...","{'Sands': ['Energy Recharge', 'ATK%', 'Element...","{1: 'Energy Recharge', 2: 'Crit Rate / DMG', 3..."
3,XIANGLING,Pyro,OFF-FIELD DPS✩,"{1: 'Staff of the Scarlet Sands (5✩)', 2: 'Lum...","{1: 'Emblem of Severed Fate (4)', 2: 'Crimson ...","{'Sands': ['Energy Recharge', 'ATK%', 'Element...","{1: 'Energy Recharge', 2: 'Crit Rate / DMG', 3..."
4,BENNETT,Pyro,SUPPORT✩,{1: 'Mistsplitter Reforged (5✩)*≈ Absolution (...,"{1: 'Noblesse Oblige (4)', 2: 'Instructor (4)*...","{'Sands': ['Energy Recharge', 'ATK%', 'HP%'], ...","{1: 'Energy Recharge', 2: 'CRIT / HP%', 3: 'AT..."
...,...,...,...,...,...,...,...
176,CHIORI,Geo,OFF-FIELD DPS✩,"{1: 'Uraku Misugiri (5✩)', 2: 'Lightbearing Mo...",{1: 'Golden Troupe (4)*≈ Husk of Opulent Dream...,"{'Sands': 'DEF%', 'Goblet': 'Geo DMG', 'Circle...","{1: 'Crit Rate / DMG', 2: 'DEF%', 3: 'ATK%', 4..."
177,XILONEN,Geo,BUFF & HEAL SUPPORT✩,"{1: 'Peak Patrol Song (5✩)*', 2: 'Freedom-Swor...","{1: 'Scroll of the Hero of Cinder City (4)', 2...","{'Sands': ['DEF%', 'Energy Recharge'], 'Goblet...","{1: 'Energy Recharge*', 2: 'DEF%', 3: 'Flat DE..."
178,ZIBAI,Geo,DPS✩,"{1: 'Lightbearing Moonshard (5✩)', 2: 'Uraku M...","{1: 'Night of the Sky's Unveiling (4)', 2: 'Hu...","{'Sands': 'DEF%', 'Goblet': 'DEF%', 'Circlet':...","{1: 'Crit Rate / DMG', 2: 'DEF%', 3: 'Energy R..."
179,LINNEA,Geo,OFF-FIELD DPS & BUFF AND HEAL SUPPORT✩,"{1: 'Golden Frostbound Oath (5✩)', 2: 'Elegy f...",{1: 'Aubade of Morningstar and Moon (4)≈ Husk ...,"{'Sands': ['DEF%', 'Energy Recharge*'], 'Goble...","{1: 'Crit Rate / DMG', 2: 'DEF%', 3: 'Elementa..."


In [8]:
df = pd.read_csv(f'gi_rsh_output_{version}.csv')

for item in ['artifacts', 'main_stats', 'substats']:
    df = normalize_df(df, item)

# Some characters have additional rows with notes about their desired stats, which manifest as additional builds.
# We filter these out by excluding builds with names that are not all upper case.
df = df[df.apply(lambda build: build['build_name'].isupper(), axis=1)]

# Add new column for prioritized builds
df['is_priority_build'] = df['build_name'].apply(lambda build_str: 1 if "✩" in str(build_str) else 0)

df

,character_name,element,build_name,weapons,artifacts_1,artifacts_2,artifacts_3,artifacts_4,artifacts_5,artifacts_6,...,artifacts_9,main_stats_Sands,main_stats_Goblet,main_stats_Circlet,substats_1,substats_2,substats_3,substats_4,substats_5,is_priority_build
0,AMBER,Pyro,MELT DPS,"{1: 'The First Great Magic (5✩)', 2: 'Aqua Sim...",Shimenawa's Reminiscence (4),Wanderer's Troupe (4),Crimson Witch of Flames (4)≈ Gilded Dreams (4),Crimson Witch of Flames (2) / +80 EM set (2) /...,NaN,NaN,...,NaN,"[ATK%, Elemental Mastery]",Pyro DMG,Crit DMG,Crit DMG,ATK%,Elemental Mastery,NaN,NaN,0
1,AMBER,Pyro,BUFF SUPPORT✩,"{1: 'Elegy for the End (5✩)', 2: 'Favonius War...",Noblesse Oblige (4),Instructor (4)*,Scroll of the Hero of Cinder City (4),The Exile (4),NaN,NaN,...,NaN,"[Energy Recharge, ATK%]",Pyro DMG,"[Crit Rate, DMG]",Energy Recharge,Crit Rate / DMG,ATK%,Elemental Mastery,NaN,1
2,XIANGLING,Pyro,OFF-FIELD VAPORIZE DPS ✩,"{1: 'Staff of the Scarlet Sands (5✩)', 2: 'Lum...",Emblem of Severed Fate (4),Crimson Witch of Flames (4)*,Gilded Dreams (4)≈ Noblesse Oblige (2) / Crims...,Noblesse Oblige (4)Conditional (See Notes):Dee...,NaN,NaN,...,NaN,"[Energy Recharge, ATK%, Elemental Mastery]",Pyro DMG,"[Crit Rate, DMG]",Energy Recharge,Crit Rate / DMG,ATK%,Elemental Mastery*,NaN,1
3,XIANGLING,Pyro,OFF-FIELD DPS✩,"{1: 'Staff of the Scarlet Sands (5✩)', 2: 'Lum...",Emblem of Severed Fate (4),Crimson Witch of Flames (4)*,Gilded Dreams (4)≈ Noblesse Oblige (2) / Crims...,Noblesse Oblige (4)Conditional (See Notes):Dee...,NaN,NaN,...,NaN,"[Energy Recharge, ATK%, Elemental Mastery]",Pyro DMG,"[Crit Rate, DMG]",Energy Recharge,Crit Rate / DMG,ATK%,Elemental Mastery*,NaN,1
4,BENNETT,Pyro,SUPPORT✩,{1: 'Mistsplitter Reforged (5✩)*≈ Absolution (...,Noblesse Oblige (4),Instructor (4)*,Scroll of the Hero of Cinder City (4)*,Deepwood Memories (4)*,NaN,NaN,...,NaN,"[Energy Recharge, ATK%, HP%]","[Pyro DMG, HP%]","[CRIT, HP%, Healing Bonus]",Energy Recharge,CRIT / HP%,ATK% / Flat HP,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,NAVIA,Geo,DPS✩,{1: 'Verdict (5✩)≈ A Thousand Blazing Suns (5✩...,Nighttime Whispers in the Echoing Woods (4),Golden Troupe (4),Archaic Petra (2) + [Choose One] +18% ATK set ...,"Marechaussee Hunter (4) *Furina teams only, pe...",NaN,NaN,...,NaN,ATK%,Geo DMG,"[Crit Rate, DMG]",Energy Recharge*,Crit Rate / DMG,ATK%*Prioritize Energy Recharge until you meet...,NaN,NaN,1
176,CHIORI,Geo,OFF-FIELD DPS✩,"{1: 'Uraku Misugiri (5✩)', 2: 'Lightbearing Mo...",Golden Troupe (4)*≈ Husk of Opulent Dreams (4)*,Golden Troupe (2) / Archaic Petra (2) / Husk o...,NaN,NaN,NaN,NaN,...,NaN,DEF%,Geo DMG,"[Crit Rate, DMG]",Crit Rate / DMG,DEF%,ATK%,Energy Recharge**Chiori is not Burst reliant a...,NaN,1
177,XILONEN,Geo,BUFF & HEAL SUPPORT✩,"{1: 'Peak Patrol Song (5✩)*', 2: 'Freedom-Swor...",Scroll of the Hero of Cinder City (4),Archaic Petra (4),Noblesse Oblige (4),Instructor (4),NaN,NaN,...,NaN,"[DEF%, Energy Recharge]",DEF%,"[DEF%, Healing Bonus, Crit Rate]",Energy Recharge*,DEF%,Flat DEF,Crit Rate***Prioritize ER until requirements a...,NaN,1
178,ZIBAI,Geo,DPS✩,"{1: 'Lightbearing Moonshard (5✩)', 2: 'Uraku M...",Night of the Sky's Unveiling (4),Husk of Opulent Dreams (4),+30% DEF set (2) / +80 EM set (2) [Choose Two],NaN,NaN,NaN,...,NaN,DEF%,DEF%,"[Crit Rate, DMG]",Crit Rate / DMG,DEF%,Energy Recharge*,Elemental Mastery*It is not realistic nor reco...,NaN,1


In [9]:
replacement_for_placeholders = {
    '80 EM set': "Wanderer's Troupe, Gilded Dreams, Flower of Paradise Lost, Aubade of Morningstar and Moon, Instructor",
    '18% ATK set': "Gladiator's Finale, Shimenawa's Reminiscence, Vermillion Hereafter, Echoes of an Offering, Nighttime Whispers in the Echoing Woods, Fragment of Harmonic Whimsy, Unfinished Reverie, A Day Carved From Rising Winds",
    '20% HP set': "Tenacity of the Millelith, Vourukasha's Glow",
    '15% Healing Bonus set': "Ocean-Hued Clam, Song of Days Past",
    '20% Energy Recharge set': "Emblem of Severed Fate, Silken Moon's Serenade, The Exile"
}

crit_patterns = [
    'Crit Rate / DMG',
    'CRIT Rate / DMG',
    'Crit Rate/DMG',
    'CRIT Rate/DMG',
    'Crit Rate, DMG',
    'CRIT Rate, DMG',
    'Crit Rate,DMG',
    'CRIT Rate,DMG',
]

crit_patterns_with_trailing_commas = [
    'Crit,',
    'CRIT,'
]

col_list = [col_name for col_name in df.columns if 'artifacts' in col_name or 'main_stats' in col_name or 'substats' in col_name]
for col in col_list:
    df[col] = df[col].apply(lambda col_value: ', '.join(col_value) if isinstance(col_value, list) else col_value)

    # Replace string variants with standardized values
    for key, value in replacement_for_placeholders.items():
        df[col] = df[col].str.replace(key, value)
    for crit_pattern in crit_patterns:
        df[col] = df[col].str.replace(crit_pattern, 'Crit Rate / Crit DMG')
    for crit_pattern in crit_patterns_with_trailing_commas:
        df[col] = df[col].str.replace(crit_pattern, 'Crit Rate / Crit DMG,')

artifact_cols = [col_name for col_name in df.columns.tolist() if 'artifacts' in col_name]
substat_cols = [col_name for col_name in df.columns.tolist() if 'substats' in col_name]

# Add new columns for spreadsheet calculations
df['substats'] = df[substat_cols].apply(lambda row: ', '.join([str(x) for x in row if pd.notna(x)]), axis=1)
df['substats'] = df['substats'].str.replace('Atk%', 'ATK%')
df['substats'] = df['substats'].str.replace('ER%', 'Energy Recharge')

df['artifacts_concat'] = '=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), COLUMN()- 4)) = 0), "", TEXTJOIN(", ", TRUE, INDIRECT(ADDRESS(ROW(), COLUMN() + 1)):INDIRECT(ADDRESS(ROW(), COLUMN() + START!$B$4))))'
df['substats_if'] = '=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), COLUMN()- 14)) = 0), "", INDIRECT(ADDRESS(ROW(), COLUMN()+1)))'

df

,character_name,element,build_name,weapons,artifacts_1,artifacts_2,artifacts_3,artifacts_4,artifacts_5,artifacts_6,...,main_stats_Circlet,substats_1,substats_2,substats_3,substats_4,substats_5,is_priority_build,substats,artifacts_concat,substats_if
0,AMBER,Pyro,MELT DPS,"{1: 'The First Great Magic (5✩)', 2: 'Aqua Sim...",Shimenawa's Reminiscence (4),Wanderer's Troupe (4),Crimson Witch of Flames (4)≈ Gilded Dreams (4),Crimson Witch of Flames (2) / +Wanderer's Trou...,NaN,NaN,...,Crit DMG,Crit DMG,ATK%,Elemental Mastery,NaN,NaN,0,"Crit DMG, ATK%, Elemental Mastery","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ..."
1,AMBER,Pyro,BUFF SUPPORT✩,"{1: 'Elegy for the End (5✩)', 2: 'Favonius War...",Noblesse Oblige (4),Instructor (4)*,Scroll of the Hero of Cinder City (4),The Exile (4),NaN,NaN,...,Crit Rate / Crit DMG,Energy Recharge,Crit Rate / Crit DMG,ATK%,Elemental Mastery,NaN,1,"Energy Recharge, Crit Rate / Crit DMG, ATK%, E...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ..."
2,XIANGLING,Pyro,OFF-FIELD VAPORIZE DPS ✩,"{1: 'Staff of the Scarlet Sands (5✩)', 2: 'Lum...",Emblem of Severed Fate (4),Crimson Witch of Flames (4)*,Gilded Dreams (4)≈ Noblesse Oblige (2) / Crims...,Noblesse Oblige (4)Conditional (See Notes):Dee...,NaN,NaN,...,Crit Rate / Crit DMG,Energy Recharge,Crit Rate / Crit DMG,ATK%,Elemental Mastery*,NaN,1,"Energy Recharge, Crit Rate / Crit DMG, ATK%, E...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ..."
3,XIANGLING,Pyro,OFF-FIELD DPS✩,"{1: 'Staff of the Scarlet Sands (5✩)', 2: 'Lum...",Emblem of Severed Fate (4),Crimson Witch of Flames (4)*,Gilded Dreams (4)≈ Noblesse Oblige (2) / Crims...,Noblesse Oblige (4)Conditional (See Notes):Dee...,NaN,NaN,...,Crit Rate / Crit DMG,Energy Recharge,Crit Rate / Crit DMG,ATK%,Elemental Mastery*,NaN,1,"Energy Recharge, Crit Rate / Crit DMG, ATK%, E...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ..."
4,BENNETT,Pyro,SUPPORT✩,{1: 'Mistsplitter Reforged (5✩)*≈ Absolution (...,Noblesse Oblige (4),Instructor (4)*,Scroll of the Hero of Cinder City (4)*,Deepwood Memories (4)*,NaN,NaN,...,"Crit Rate / Crit DMG, HP%, Healing Bonus",Energy Recharge,CRIT / HP%,ATK% / Flat HP,NaN,NaN,1,"Energy Recharge, CRIT / HP%, ATK% / Flat HP","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,NAVIA,Geo,DPS✩,{1: 'Verdict (5✩)≈ A Thousand Blazing Suns (5✩...,Nighttime Whispers in the Echoing Woods (4),Golden Troupe (4),Archaic Petra (2) + [Choose One] +Gladiator's ...,"Marechaussee Hunter (4) *Furina teams only, pe...",NaN,NaN,...,Crit Rate / Crit DMG,Energy Recharge*,Crit Rate / Crit DMG,ATK%*Prioritize Energy Recharge until you meet...,NaN,NaN,1,"Energy Recharge*, Crit Rate / Crit DMG, ATK%*P...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ..."
176,CHIORI,Geo,OFF-FIELD DPS✩,"{1: 'Uraku Misugiri (5✩)', 2: 'Lightbearing Mo...",Golden Troupe (4)*≈ Husk of Opulent Dreams (4)*,Golden Troupe (2) / Archaic Petra (2) / Husk o...,NaN,NaN,NaN,NaN,...,Crit Rate / Crit DMG,Crit Rate / Crit DMG,DEF%,ATK%,Energy Recharge**Chiori is not Burst reliant a...,NaN,1,"Crit Rate / Crit DMG, DEF%, ATK%, Energy Recha...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ..."
177,XILONEN,Geo,BUFF & HEAL SUPPORT✩,"{1: 'Peak Patrol Song (5✩)*', 2: 'Freedom-Swor...",Scroll of the Hero of Cinder City (4),Archaic Petra (4),Noblesse Oblige (4),Instructor (4),NaN,NaN,...,"DEF%, Healing Bonus, Crit Rate",Energy Recharge*,DEF%,Flat DEF,Crit Rate***Prioritize ER until requirements a...,NaN,1,"Energy Recharge*, DEF%, Flat DEF, Crit Rate***...","=IF(AND(START!$B$3=1, INDIRECT(ADDRE

In [10]:
# Reorder columns
new_col_order = ['character_name', 'element', 'build_name', 'is_priority_build', 'main_stats_Sands', 'main_stats_Goblet',
                 'main_stats_Circlet', 'artifacts_concat']
new_col_order.extend(artifact_cols)
new_col_order.append('substats_if')
new_col_order.append('substats')
new_col_order.extend(substat_cols)

df = df[new_col_order]
df

,character_name,element,build_name,is_priority_build,main_stats_Sands,main_stats_Goblet,main_stats_Circlet,artifacts_concat,artifacts_1,artifacts_2,...,artifacts_7,artifacts_8,artifacts_9,substats_if,substats,substats_1,substats_2,substats_3,substats_4,substats_5
0,AMBER,Pyro,MELT DPS,0,"ATK%, Elemental Mastery",Pyro DMG,Crit DMG,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...",Shimenawa's Reminiscence (4),Wanderer's Troupe (4),...,NaN,NaN,NaN,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","Crit DMG, ATK%, Elemental Mastery",Crit DMG,ATK%,Elemental Mastery,NaN,NaN
1,AMBER,Pyro,BUFF SUPPORT✩,1,"Energy Recharge, ATK%",Pyro DMG,Crit Rate / Crit DMG,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...",Noblesse Oblige (4),Instructor (4)*,...,NaN,NaN,NaN,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","Energy Recharge, Crit Rate / Crit DMG, ATK%, E...",Energy Recharge,Crit Rate / Crit DMG,ATK%,Elemental Mastery,NaN
2,XIANGLING,Pyro,OFF-FIELD VAPORIZE DPS ✩,1,"Energy Recharge, ATK%, Elemental Mastery",Pyro DMG,Crit Rate / Crit DMG,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...",Emblem of Severed Fate (4),Crimson Witch of Flames (4)*,...,NaN,NaN,NaN,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","Energy Recharge, Crit Rate / Crit DMG, ATK%, E...",Energy Recharge,Crit Rate / Crit DMG,ATK%,Elemental Mastery*,NaN
3,XIANGLING,Pyro,OFF-FIELD DPS✩,1,"Energy Recharge, ATK%, Elemental Mastery",Pyro DMG,Crit Rate / Crit DMG,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...",Emblem of Severed Fate (4),Crimson Witch of Flames (4)*,...,NaN,NaN,NaN,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","Energy Recharge, Crit Rate / Crit DMG, ATK%, E...",Energy Recharge,Crit Rate / Crit DMG,ATK%,Elemental Mastery*,NaN
4,BENNETT,Pyro,SUPPORT✩,1,"Energy Recharge, ATK%, HP%","Pyro DMG, HP%","Crit Rate / Crit DMG, HP%, Healing Bonus","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...",Noblesse Oblige (4),Instructor (4)*,...,NaN,NaN,NaN,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","Energy Recharge, CRIT / HP%, ATK% / Flat HP",Energy Recharge,CRIT / HP%,ATK% / Flat HP,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,NAVIA,Geo,DPS✩,1,ATK%,Geo DMG,Crit Rate / Crit DMG,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...",Nighttime Whispers in the Echoing Woods (4),Golden Troupe (4),...,NaN,NaN,NaN,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","Energy Recharge*, Crit Rate / Crit DMG, ATK%*P...",Energy Recharge*,Crit Rate / Crit DMG,ATK%*Prioritize Energy Recharge until you meet...,NaN,NaN
176,CHIORI,Geo,OFF-FIELD DPS✩,1,DEF%,Geo DMG,Crit Rate / Crit DMG,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...",Golden Troupe (4)*≈ Husk of Opulent Dreams (4)*,Golden Troupe (2) / Archaic Petra (2) / Husk o...,...,NaN,NaN,NaN,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","Crit Rate / Crit DMG, DEF%, ATK%, Energy Recha...",Crit Rate / Crit DMG,DEF%,ATK%,Energy Recharge**Chiori is not Burst reliant a...,NaN
177,XILONEN,Geo,BUFF & HEAL SUPPORT✩,1,"DEF%, Energy Recharge",DEF%,"DEF%, Healing Bonus, Crit Rate","=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...",Scroll of the Hero of Cinder City (4),Archaic Petra (4),...,NaN,NaN,NaN,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","Energy Recharge*, DEF%, Flat DEF, Crit Rate***...",Energy Recharge*,DEF%,Flat DEF,Crit Rate***Prioritize ER until requirements a...,NaN
178,ZIBAI,Geo,DPS✩,1,DEF%,DEF%,Crit Rate / Crit DMG,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...",Night of the Sky's Unveiling (4),Husk of Opulent Dreams (4),...,NaN,NaN,NaN,"=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), ...","Crit Rate / Crit DMG, DEF%, Energy Recharge*, ...",Crit Rate / Crit DMG,DEF%,Energy Recharge*,Elemental Mastery*It is not realistic nor reco...,NaN


In [11]:
# Export final csv
df.to_csv(f'gi_rsh_output_full_{version}.csv', index=False)